# Experiment 09 — Stress Test (N=5000)

**Goal:** Re-evaluate the comparison between the Classical MLP and the Optimized VQC using 5,000 samples (the original paper's volume).

**Setup:**
- **Data:** 5,000 samples, 8 features, leak-free pipeline.
- **Loss:** Binary Cross-Entropy (BCE) for both models.
- **VQC:** 8 qubits, 3 layers, Data Re-uploading, Circular Entanglement.

In [1]:
import sys
sys.path.append('..')
import time
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from utils.data_utils import load_higgs

N_SAMPLES = 5000
N_FEATURES = 8
PATH = '../data/HIGGS.csv.gz'
SEED = 42

## 1. Classical MLP Benchmark (N=5000)

In [2]:
X_tr_c, X_vl_c, X_te_c, y_tr_c, y_vl_c, y_te_c = load_higgs(
    path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=SEED, scale_range=(0, 1)
)

mlp = MLPClassifier(hidden_layer_sizes=(100, 50, 25), max_iter=500, early_stopping=True, random_state=SEED)
mlp.fit(X_tr_c, y_tr_c)

c_auc = roc_auc_score(y_te_c, mlp.predict_proba(X_te_c)[:, 1])
print(f"Classical Test AUC (N=5000): {c_auc:.4f}")

Dataset: 5000 samples | 8 features
Split: train=3000, val=1000, test=1000
Classical Test AUC (N=5000): 0.6782


## 2. Optimized VQC Benchmark (N=5000)

In [3]:
dev = qml.device('lightning.qubit', wires=N_FEATURES)

@qml.qnode(dev, interface='autograd')
def circuit(weights, x):
    for l in range(3):
        for i in range(N_FEATURES): qml.RY(x[i], wires=i)
        for q in range(N_FEATURES): qml.Rot(*weights[l, q], wires=q)
        for q in range(N_FEATURES): qml.CNOT(wires=[q, (q + 1) % N_FEATURES])
    return qml.expval(qml.PauliZ(0))

def vqc_prob(w, x, b):
    return (circuit(w, x) + b + 1.0) / 2.0

def bce_loss(weights, bias, X, y):
    # We loop for safety with autograd broadcast constraints
    probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
    probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
    return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))

def train_quantum_5000():
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=SEED, scale_range=(0, np.pi)
    )
    
    pnp.random.seed(SEED)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (3, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    print("Starting Quantum Training (N=5000)...")
    for epoch in range(30):
        idx = np.random.permutation(len(X_tr))
        for start in range(0, len(X_tr), 128):
            Xb, yb = X_tr[idx][start:start+128], y_tr[idx][start:start+128].astype(float)
            w, b = opt.step(lambda w_, b_: bce_loss(w_, b_, Xb, yb), w, b)
        print(f"  Epoch {epoch+1} complete")
        
    q_probs = np.array([float(vqc_prob(w, x, b)) for x in X_te])
    q_auc = roc_auc_score(y_te, q_probs)
    return q_auc

q_auc = train_quantum_5000()
print(f"\nQuantum Test AUC (N=5000): {q_auc:.4f}")

Dataset: 5000 samples | 8 features
Split: train=3000, val=1000, test=1000
Starting Quantum Training (N=5000)...
  Epoch 1 complete
  Epoch 2 complete
  Epoch 3 complete
  Epoch 4 complete
  Epoch 5 complete
  Epoch 6 complete
  Epoch 7 complete
  Epoch 8 complete
  Epoch 9 complete
  Epoch 10 complete
  Epoch 11 complete
  Epoch 12 complete
  Epoch 13 complete
  Epoch 14 complete
  Epoch 15 complete
  Epoch 16 complete
  Epoch 17 complete
  Epoch 18 complete
  Epoch 19 complete
  Epoch 20 complete
  Epoch 21 complete
  Epoch 22 complete
  Epoch 23 complete
  Epoch 24 complete
  Epoch 25 complete
  Epoch 26 complete
  Epoch 27 complete
  Epoch 28 complete
  Epoch 29 complete
  Epoch 30 complete

Quantum Test AUC (N=5000): 0.6719


## 3. Final Conclusion

In [4]:
print(f"--- RESULTS (N=5000) ---")
print(f"Classical AUC: {c_auc:.4f}")
print(f"Quantum AUC:   {q_auc:.4f}")
print(f"Advantage:     {q_auc - c_auc:.4f}")

--- RESULTS (N=5000) ---
Classical AUC: 0.6782
Quantum AUC:   0.6719
Advantage:     -0.0062
